# News Headline Classification with BERT

This notebook fine-tunes `bert-base-uncased` on the AG News dataset from Hugging Face Datasets. It tokenizes and preprocesses news headlines, trains a transformer classifier, evaluates accuracy and F1-score, and saves the model for Streamlit deployment.

In [ ]:

# first i have created an env and installed pip install torch transformers datasets evaluate scikit-learn gradio accelerate pandas numpy al these libraries are required for this code to run after that i have installed ipykernel
import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)
from sklearn.metrics import accuracy_score, f1_score

plt = None

model_dir = "./news_classifier_model"

# Load AG News dataset
raw_datasets = load_dataset("ag_news")
label_names = raw_datasets["train"].features["label"].names
print("Label names:", label_names)
print(raw_datasets)

raw_datasets["train"].shuffle(seed=42).select(range(5)).to_pandas()

In [ ]:
import sys
print(sys.executable)

In [ ]:
# Tokenize and preprocess
checkpoint = "bert-base-uncased"
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )

processed_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"],
)
processed_datasets = processed_datasets.rename_column("label", "labels")
processed_datasets.set_format("torch")

# Use a subset for faster local fine-tuning; remove select() to train on the full dataset.
train_dataset = processed_datasets["train"].shuffle(seed=42).select(range(20000))
eval_dataset = processed_datasets["test"].shuffle(seed=42).select(range(5000))

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

In [ ]:
# Build model and trainer
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=len(label_names))
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
    }

training_args = TrainingArguments(
    output_dir=model_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

metrics = trainer.evaluate()
print(metrics)

trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

In [ ]:
# Evaluate on additional held-out examples and inspect predictions
classifier = pipeline(
    "text-classification",
    model=model_dir,
    tokenizer=model_dir,
    return_all_scores=True,
)

examples = [
    "Stocks surged after the latest quarterly earnings report.",
    "The championship game ended with a dramatic last-minute goal.",
    "The new phone release pushed the company shares higher.",
    "Scientists announced a breakthrough in quantum computing."
]

for text in examples:
    results = classifier(text)[0]
    label_scores = {item["label"]: item["score"] for item in results}
    mapped = {
        label_names[int(label.split("_")[-1])]: score
        for label, score in label_scores.items()
    }
    print(text)
    print(mapped)
    print()